# Part 2 — Fine-Tuning ESM-2 on TEM-1 DMS Data

In Part 1, we used ESM-2 zero-shot. The signal was real (Spearman ρ ≈ 0.5) but uncalibrated, and the small 35M model missed some biologically obvious mutations like W165R. Now we'll **fine-tune** the model on a subset of the Firnberg deep mutational scanning data and see how much it improves.

**You will:**
1. Prepare the Firnberg DMS dataset with proper position-based splits (no homology leakage)
2. Add a regression head and apply LoRA for parameter-efficient fine-tuning
3. Train the model
4. Compare fine-tuned vs zero-shot predictions on held-out mutations

**Runtime:** ~30 min on a Colab T4 GPU. ~3 hours on a laptop CPU/MPS.

> Make sure you've completed [Part 1](./01_zero_shot_scoring.ipynb) first — it explains the biology, the numbering conventions, and the zero-shot baseline we're comparing against.

## 1. Setup

In addition to Part 1's dependencies, we need `datasets`, `peft` (for LoRA), and `accelerate`.

In [ ]:
# Standard library
import re

# Third-party libraries
# Note: On Colab, uncomment the next line:
# !pip install transformers datasets peft accelerate scipy scikit-learn matplotlib pandas requests -q
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error
from transformers import (
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [ ]:
# Device detection (works on CPU, CUDA, or Apple Silicon MPS)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

## 2. The TEM-1 sequence

Same setup as Part 1: mature sequence from UniProt P62593, with Ambler numbering helpers.

In [ ]:
def fetch_tem1_sequence():
    """Fetch the canonical TEM-1 mature sequence from UniProt (P62593, residues 24-286)."""
    url = "https://rest.uniprot.org/uniprotkb/P62593.fasta"
    fasta = requests.get(url).text
    return "".join(fasta.split("\n")[1:])


TEM1_FULL = fetch_tem1_sequence()
assert len(TEM1_FULL) == 286, "Full sequence length is not 286"

SIGNAL_PEPTIDE_LENGTH = 23
TEM1_MATURE = TEM1_FULL[SIGNAL_PEPTIDE_LENGTH:]
assert len(TEM1_MATURE) == 263, "Mature sequence length is not 263"

# Numbering helpers (see Part 1 for explanation)
AMBLER_TO_UNIPROT_OFFSET = 2


def uniprot_to_index(pos):
    """UniProt position (1-indexed on full precursor) -> 0-indexed on mature sequence."""
    return pos - SIGNAL_PEPTIDE_LENGTH - 1


def ambler_to_uniprot(pos):
    """Convert Ambler position to the equivalent UniProt position (both 1-indexed)."""
    return pos - AMBLER_TO_UNIPROT_OFFSET


def ambler_to_index(pos):
    """Convert Ambler position to 0-indexed position in the mature sequence."""
    return uniprot_to_index(ambler_to_uniprot(pos))


# Sanity check
assert TEM1_MATURE[ambler_to_index(70)] == "S", "Catalytic S70 check failed"
print(f"Mature sequence: {len(TEM1_MATURE)} residues ✓")

## 3. Load and split the Firnberg dataset

A subtle but **critical** point: how we split the data matters enormously.

- **Random splitting** would put many mutations at the same position into both train and test sets. The model could memorize "position X is hard" rather than learning generalizable features.
- **Position-based splitting** holds out entire positions. The model has to predict effects at residues it has never seen mutated during training.

Position-based splitting gives a more honest measure of generalization, and it's the standard approach for DMS benchmarks.

In [ ]:
# File mirrored in this repo from ProteinGym v1.3
# Original source: https://github.com/OATML-Markslab/ProteinGym
dms = pd.read_csv("data/BLAT_ECOLX_Firnberg_2014.csv")
print(f"Loaded {len(dms)} mutations")


# Parse the mutant string and convert ProteinGym (UniProt) positions to Ambler
def parse_mutant(s):
    m = re.match(r"([A-Z])(\d+)([A-Z])", s)
    if not m:
        return None
    return m.group(1), int(m.group(2)), m.group(3)


dms[["wt", "pos", "mut"]] = dms["mutant"].apply(parse_mutant).tolist()
dms["ambler_pos"] = dms["pos"] + AMBLER_TO_UNIPROT_OFFSET
dms = dms.dropna(subset=["wt", "pos", "mut"]).reset_index(drop=True)


# Filter to mutations that fall within the mature sequence and match our reference
def is_valid(row):
    idx = ambler_to_index(row["ambler_pos"])
    if idx < 0 or idx >= len(TEM1_MATURE):
        return False
    return TEM1_MATURE[idx] == row["wt"]


dms["valid"] = dms.apply(is_valid, axis=1)
dms = dms[dms["valid"]].reset_index(drop=True)
print(f"After filtering to valid single-position mutations: {len(dms)}")

In [ ]:
# Position-based split: 70% train, 15% valid, 15% test
unique_positions = sorted(dms["ambler_pos"].unique())
rng = np.random.default_rng(42)
shuffled = rng.permutation(unique_positions)
n = len(shuffled)
train_pos = set(shuffled[: int(0.7 * n)])
val_pos = set(shuffled[int(0.7 * n) : int(0.85 * n)])
test_pos = set(shuffled[int(0.85 * n) :])

train_df = dms[dms["ambler_pos"].isin(train_pos)].reset_index(drop=True)
val_df = dms[dms["ambler_pos"].isin(val_pos)].reset_index(drop=True)
test_df = dms[dms["ambler_pos"].isin(test_pos)].reset_index(drop=True)

print(f"Train: {len(train_df)} mutations at {len(train_pos)} positions")
print(f"Valid: {len(val_df)} mutations at {len(val_pos)} positions")
print(f"Test:  {len(test_df)} mutations at {len(test_pos)} positions")

For each mutation, we construct the mutated mature sequence and use the experimental `DMS_score` as the regression target.

In [ ]:
def build_mutated_sequence(row):
    """Apply the single-point mutation to the mature TEM-1 sequence."""
    idx = ambler_to_index(row["ambler_pos"])
    return TEM1_MATURE[:idx] + row["mut"] + TEM1_MATURE[idx + 1 :]


for df in (train_df, val_df, test_df):
    df["mutated_sequence"] = df.apply(build_mutated_sequence, axis=1)

print(f"Example: mutation {train_df.iloc[0]['mutant']} (Ambler {train_df.iloc[0]['ambler_pos']})")
print(f"  WT seq[:20]:  {TEM1_MATURE[:20]}…")
print(f"  Mut seq[:20]: {train_df.iloc[0]['mutated_sequence'][:20]}…")

In [ ]:
# Standardize the regression target using TRAIN statistics only.
# Random init of the regression head outputs values near zero, so if targets
# are centered far from zero (or have large variance), early gradients are
# dominated by the bias term and the model collapses to predicting the mean.
# Standardizing forces meaningful weight learning from step 1.
TARGET_MEAN = train_df["DMS_score"].mean()
TARGET_STD = train_df["DMS_score"].std()
print(f"Target mean (train): {TARGET_MEAN:.3f}")
print(f"Target std  (train): {TARGET_STD:.3f}")

for df in (train_df, val_df, test_df):
    df["DMS_score_std"] = (df["DMS_score"] - TARGET_MEAN) / TARGET_STD

## 4. Tokenize and build datasets

In [ ]:
MODEL_NAME = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf_dataset(df):
    return Dataset.from_pandas(
        df[["mutated_sequence", "DMS_score_std"]].rename(
            columns={"mutated_sequence": "sequence", "DMS_score_std": "labels"}
        ),
        preserve_index=False,
    )


def tokenize(batch):
    enc = tokenizer(batch["sequence"], truncation=True, max_length=300, padding=False)
    enc["labels"] = [float(x) for x in batch["labels"]]
    return enc


train_ds = to_hf_dataset(train_df).map(tokenize, batched=True, remove_columns=["sequence"])
val_ds = to_hf_dataset(val_df).map(tokenize, batched=True, remove_columns=["sequence"])
test_ds = to_hf_dataset(test_df).map(tokenize, batched=True, remove_columns=["sequence"])

print(train_ds)

## 5. Build the model: ESM-2 + regression head + LoRA

We attach a regression head on top of pretrained ESM-2 (`num_labels=1` + `problem_type="regression"`), then apply **LoRA** to fine-tune efficiently.

**LoRA in one paragraph:** instead of updating all 35M parameters, we freeze the base model and inject small trainable matrices into the attention layers. Typically <1% of parameters become trainable, which:
- Drastically reduces memory usage (important on laptops)
- Helps prevent overfitting on small datasets
- Trains much faster
- Loses surprisingly little performance vs full fine-tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression"
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,  # 8 → 16 (or even 32)
    lora_alpha=32,  # keep alpha = 2 × r
    lora_dropout=0.05,
    target_modules=["query", "key", "value", "dense"],  # ESM attention projections
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import torch

# Take a few mutant sequences and look at their CLS-token embeddings
mlm_model_eval = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()

with torch.no_grad():
    seqs = train_df["mutated_sequence"].head(5).tolist()
    inputs = tokenizer(seqs, return_tensors="pt", padding=True).to(device)
    # Grab the encoder hidden states (not the LM head)
    out = mlm_model_eval.esm(**inputs)
    hidden = out.last_hidden_state  # [batch, seqlen, 480]

    # CLS-token embedding
    cls_emb = hidden[:, 0, :]
    # Mean-pooled embedding (over actual residues, masking padding)
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    mean_emb = (hidden * mask).sum(1) / mask.sum(1)

    print(f"CLS embedding variance across 5 sequences:  {cls_emb.var(0).mean().item():.4f}")
    print(f"Mean-pool variance across 5 sequences:      {mean_emb.var(0).mean().item():.4f}")

In [ ]:
from sklearn.linear_model import Ridge


# Extract mean-pooled embeddings for all train/test sequences
@torch.no_grad()
def get_embedding(seq):
    inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=300).to(device)
    out = mlm_model_eval.esm(**inputs)
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    return ((out.last_hidden_state * mask).sum(1) / mask.sum(1)).cpu().numpy()[0]


print("Computing embeddings (this takes a few minutes)...")
X_train = np.stack([get_embedding(s) for s in train_df["mutated_sequence"]])
X_test = np.stack([get_embedding(s) for s in test_df["mutated_sequence"]])
y_train = train_df["DMS_score"].values
y_test = test_df["DMS_score"].values

ridge = Ridge(alpha=1.0).fit(X_train, y_train)
pred = ridge.predict(X_test)
rho = spearmanr(pred, y_test).correlation
print(f"Ridge on ESM-2 embeddings: Spearman ρ = {rho:.3f}")

## 6. Train

Settings tuned for a small dataset on modest hardware. Key choices:

- **Learning rate `1e-4`** — higher than typical full fine-tuning because LoRA tolerates larger steps
- **Effective batch size 16** (via gradient accumulation) — reasonable for stable optimization
- **Early stopping on validation Spearman ρ** — avoids overfitting

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()
    return {
        "spearman": spearmanr(preds, labels).correlation,
        "mae": mean_absolute_error(labels, preds),
        "rmse": float(np.sqrt(((preds - labels) ** 2).mean())),
    }


# Adjust batch size based on hardware
bs_train = 4 if device != "cpu" else 2
grad_accum = 16 // bs_train

args = TrainingArguments(
    output_dir="tem1_finetune",
    learning_rate=3e-4,  # 1e-4 → 3e-4
    per_device_train_batch_size=bs_train,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=10,  # 5 → 10    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    fp16=(device == "cuda"),  # MPS doesn't support fp16 well
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 7. Evaluate on the held-out test set

In [ ]:
test_metrics = trainer.evaluate(test_ds)
print("Test set:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.3f}")

## 8. Zero-shot vs fine-tuned: head-to-head

Now the key comparison. We re-score the test set with the zero-shot ESM-2 (using the same masked marginal likelihood approach as Part 1) and compare against the fine-tuned predictions on the same mutations.

In [ ]:
# Load a separate copy of ESM-2 for zero-shot scoring (the fine-tuned one has a regression head)
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()

AA = list("ACDEFGHIKLMNPQRSTVWY")
aa_token_ids = [tokenizer.convert_tokens_to_ids(a) for a in AA]


@torch.no_grad()
def zero_shot_score(seq, position_0idx, mut_aa):
    """Masked marginal log-likelihood ratio (same as Part 1)."""
    masked = seq[:position_0idx] + tokenizer.mask_token + seq[position_0idx + 1 :]
    inputs = tokenizer(masked, return_tensors="pt").to(device)
    logits = mlm_model(**inputs).logits
    mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero()[0, 1].item()
    log_probs = torch.log_softmax(logits[0, mask_idx, aa_token_ids], dim=-1)
    return (log_probs[AA.index(mut_aa)] - log_probs[AA.index(seq[position_0idx])]).item()


# Score every test mutation zero-shot
zs_scores = []
exp_scores = []
for _, row in test_df.iterrows():
    idx = ambler_to_index(row["ambler_pos"])
    s = zero_shot_score(TEM1_MATURE, idx, row["mut"])
    zs_scores.append(s)
    exp_scores.append(row["DMS_score"])

zs_rho = spearmanr(zs_scores, exp_scores).correlation
ft_rho = test_metrics["eval_spearman"]

print(f"Zero-shot Spearman ρ:    {zs_rho:.3f}")
print(f"Fine-tuned Spearman ρ:   {ft_rho:.3f}")
print(f"Improvement:             {ft_rho - zs_rho:+.3f}")

In [ ]:
# What does the trainer evaluate to right now?
print(trainer.evaluate(test_ds))

In [ ]:
# What's actually in the training log?
for log in trainer.state.log_history:
    print(log)

In [ ]:
# Check what's actually trainable
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"trainable: {name}  shape={tuple(param.shape)}")

In [ ]:
print(f"Test set size: {len(test_df)}")

In [ ]:
# Side-by-side scatter plots
ft_preds = trainer.predict(test_ds).predictions.squeeze()

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

axes[0].scatter(zs_scores, exp_scores, alpha=0.4, s=14, color="steelblue")
axes[0].set_xlabel("Zero-shot ESM-2 score")
axes[0].set_ylabel("Experimental fitness")
axes[0].set_title(f"Zero-shot — ρ = {zs_rho:.3f}")
axes[0].axhline(0, color="grey", lw=0.5)
axes[0].axvline(0, color="grey", lw=0.5)

axes[1].scatter(
    ft_preds, test_df["DMS_score"].values[: len(ft_preds)], alpha=0.4, s=14, color="firebrick"
)
axes[1].set_xlabel("Fine-tuned ESM-2 prediction")
axes[1].set_title(f"Fine-tuned — ρ = {ft_rho:.3f}")
axes[1].axhline(0, color="grey", lw=0.5)

plt.tight_layout()
plt.show()

**What you should see:**

- The fine-tuned scatter is tighter around the diagonal — the model has learned the specific scale and distribution of the Firnberg fitness scores
- Zero-shot has more scatter but still a clear positive trend — the pretraining signal is real
- A typical improvement is +0.10 to +0.20 Spearman ρ on TEM-1 with the 35M model

## 9. Did fine-tuning fix W165R?

Recall from Part 1: zero-shot ESM-2 35M scored **W165R as +0.95** — wrongly predicting that this destabilizing mutation is favorable. Did fine-tuning catch it?

Note: this only works if W165R happens to be in the test set. If it's in train or validation, the fine-tuned prediction would be biased.

In [ ]:
target_mutations = ["M182T", "G238S", "E104K", "A42G", "W165R", "R65L"]

print(f"{'Mutation':<10}{'Split':<8}{'Exp':<10}{'Zero-shot':<12}{'Fine-tuned':<12}")
print("-" * 55)
for mut_name in target_mutations:
    m = re.match(r"([A-Z])(\d+)([A-Z])", mut_name)
    wt, ambler_pos, target_aa = m.group(1), int(m.group(2)), m.group(3)

    # Find which split it's in
    pos_set_label = (
        "train"
        if ambler_pos in train_pos
        else "valid"
        if ambler_pos in val_pos
        else "test"
        if ambler_pos in test_pos
        else "?"
    )

    # Look up experimental score
    matching = dms[(dms["ambler_pos"] == ambler_pos) & (dms["mut"] == target_aa)]
    if len(matching) == 0:
        exp = "missing"
    else:
        exp = f"{matching.iloc[0]['DMS_score']:.2f}"

    # Zero-shot score (always available)
    idx = ambler_to_index(ambler_pos)
    if 0 <= idx < len(TEM1_MATURE) and TEM1_MATURE[idx] == wt:
        zs = f"{zero_shot_score(TEM1_MATURE, idx, target_aa):+.2f}"
    else:
        zs = "n/a"

    # Fine-tuned prediction (only meaningful for test set)
    if pos_set_label == "test" and len(matching) > 0:
        # Build the mutated sequence and run inference
        mut_seq = TEM1_MATURE[:idx] + target_aa + TEM1_MATURE[idx + 1 :]
        inputs = tokenizer(mut_seq, return_tensors="pt", truncation=True, max_length=300).to(device)
        with torch.no_grad():
            ft_pred = model(**inputs).logits.item()
        ft = f"{ft_pred:+.2f}"
    else:
        ft = "(not test)"

    print(f"{mut_name:<10}{pos_set_label:<8}{exp:<10}{zs:<12}{ft:<12}")

**Interpretation:**

- If W165R is in the test set, you'll see whether fine-tuning fixed the zero-shot error
- If it's in train/valid, the fine-tuned prediction would be optimistic — the model has seen position 165 during training
- This is exactly why position-based splitting matters: it forces the model to generalize to unseen positions, not memorize them

The general lesson: **fine-tuning corrects systematic errors that zero-shot makes**, particularly for proteins where the experimental conditions or specific structural context isn't well-captured by general evolutionary patterns.

## 10. Discussion: when is fine-tuning worth it?

**Fine-tuning helped because:**
- We had thousands of training examples — well above the threshold where fine-tuning typically beats simpler approaches
- The DMS data is relatively clean and consistent (single experimental setup)
- TEM-1 is well within ESM-2's training distribution

**The cost of fine-tuning:**
- The model is now **specialized to TEM-1 under Firnberg's experimental conditions**. It won't transfer well to:
  - Other β-lactamases (would need cross-protein training data)
  - Different antibiotics (the assay-specific fitness signal would change)
  - Pure stability prediction (fitness ≠ stability — it's a composite of stability, activity, and expression)
- Compute cost: a one-time training run, but you may need to repeat it as you collect more data

**A useful decision tree:**

| Situation | Recommended approach |
|---|---|
| No labeled data | Zero-shot pseudo-likelihood (Part 1) |
| <100 examples | Frozen embeddings + ridge regression |
| 100–10,000 examples | LoRA fine-tuning (this notebook) |
| >10,000 examples | Full fine-tuning, possibly larger ESM-2 |
| Need cross-protein generalization | Train on ProTherm/FireProt instead of single-protein DMS |
| Need 3D-aware predictions | Try SaProt or AlphaFold-derived features |

## 11. Where to go next

You now have a complete pipeline for protein mutation effect prediction. Some natural extensions:

- **Frozen embeddings baseline:** extract ESM-2 embeddings once, then train ridge regression or a small MLP on top. Often within a few percent of full fine-tuning at <1% of the compute.
- **Try a bigger model:** swap `esm2_t12_35M_UR50D` for `esm2_t33_650M_UR50D`. Expect +5 to +10 Spearman points on a GPU.
- **Different protein, same recipe:** ProteinGym has DMS data for ~80 proteins. The code here works on any of them with minimal changes.
- **Multi-task learning:** train one model that predicts stability, activity, and expression jointly.
- **Structural integration:** try SaProt or ESM-3 to add structure-aware features.

**Acknowledgments and citations** are in the [README](./README.md). Issues and pull requests welcome.